### Cell 01 - Import Dependencies and Basic Settings

This cell imports libraries for numerical computing, plotting, table I/O, and GP-HT-related routines, and sets basic paths or plotting styles used later.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

- Function descriptions: 
  - `ensure_dir(path)`: Takes a directory path, creates missing directories, and returns the same `Path` object for later table or figure output.


In [ ]:
# Import the core libraries required by this notebook
import os
import re
from pathlib import Path
from math import pi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
# GP_hilbert.py is imported from the current folder. It is based on the companion code of the original GP-HT work by Ciucci et al. (J. Electrochem. Soc. 2020, DOI: 10.1149/1945-7111/aba9c0).
import GP_hilbert as gpf
# Set plotting fonts and axis rendering so figures display consistently in Windows and JupyterLab
plt.rc('font', family='serif', size=13)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=5, suppress=True)
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


### Cell 02 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.
- Main parameters: MODEL_NAME, OUTPUT_DIR, DATA_DIR, FIG_DIR, REG_FIG_DIR, GPHT_FIG_DIR, DEG_FIG_DIR, RANDOM_SEED, N_FREQS, FREQ_RANGE, PRED_RANGE, N_PRED, NOISE_LEVELS, SPARSE_RATIOS, LIMITED_RANGE_LIST, LIMITED_PERCENT_LIST, GPHT_CONFIG.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.

- Function descriptions:
  - `ensure_dir(path)`: Takes a directory path, creates missing directories, and returns the same `Path` object for later table or figure output.
- `GPHT_CONFIG` parameter description: `SIGMA_DRT_FACTOR` controls the DRT-kernel amplitude; `THETA_SIGMA_DRT_FACTOR` sets the optimization initial-value ratio; `SIGMA_DRT_LOWER_FACTOR` and `SIGMA_DRT_UPPER_FACTOR` set the DRT-kernel amplitude bounds; `SIGMA_SB_INIT_FACTOR` and `SIGMA_SB_WEAK_FACTOR` control the boundary or smooth-background kernel; `ELL_INIT`, `ELL_LOWER`, and `ELL_UPPER` control the background-kernel length scale; `SIGMA_N_FLOOR_FACTOR` settings define the observation-noise floor to reduce overfitting; `TAU_MAX` settings control the longest DRT relaxation time represented by the kernel.

In [ ]:
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path
MODEL_NAME = 'DoubleCole'
OUTPUT_DIR = ensure_dir(Path(r'C:\Users\CYJ\Desktop\GP-HT_DoubleCole_case_V2'))  # This is the author's local input/output path; please update it before running.
DATA_DIR = ensure_dir(OUTPUT_DIR / 'data')  # This is the author's local input/output path; please update it before running.
FIG_DIR = ensure_dir(OUTPUT_DIR / 'figures')  # This is the author's local input/output path; please update it before running.
REG_FIG_DIR = ensure_dir(FIG_DIR / 'regression')  # This is the author's local input/output path; please update it before running.
GPHT_FIG_DIR = ensure_dir(FIG_DIR / 'dual_GPHT')  # This is the author's local input/output path; please update it before running.
DEG_FIG_DIR = ensure_dir(FIG_DIR / 'degraded')  # This is the author's local input/output path; please update it before running.
RANDOM_SEED = 20260614
# Frequency range and number of points
N_FREQS = 200
FREQ_RANGE = (1e1, 1e7)
PRED_RANGE = (1e1, 1e7)
N_PRED = 200
NOISE_LEVELS = [0.01, 0.10, 0.20]  # Multiplicative relative noise levels to generate; 0.10 means Z_new = Z * (1 + N(0, 0.10)).
SPARSE_RATIOS = [3,5,10]  # Sparse-sampling ratio; 3 means keeping one point every three frequency points.
LIMITED_RANGE_LIST = [(5e1, 1e6), (1e2, 1e6)]  # Frequency-range retention settings; each tuple is (minimum frequency, maximum frequency), and an empty list disables this condition.
LIMITED_PERCENT_LIST = [0.20, 0.30, 0.40]  # Point-count truncation settings; 0.20 removes 10% of points from each low- and high-frequency end.
# Sparse-sampling condition settings
GPHT_CONFIG = {
    'SIGMA_DRT_FACTOR': 0.20,  # Amplitude of the DRT kernel relative to the current impedance scale; controls the overall prior amplitude of the relaxation distribution.
    'THETA_SIGMA_DRT_FACTOR': 1.0,  # Initial ratio for the sigma_DRT optimization variable; affects the starting point of hyperparameter optimization.
    'SIGMA_DRT_LOWER_FACTOR': 1e-5,  # Relative lower bound for sigma_DRT; prevents the DRT-kernel amplitude from collapsing to zero.
    'SIGMA_DRT_UPPER_FACTOR': 50.0,  # Relative upper bound for sigma_DRT; limits the relaxation-kernel amplitude to reduce over-oscillation.
    'SIGMA_SB_INIT_FACTOR': 0.01,  # Initial amplitude ratio for the boundary/smooth-background kernel; used to absorb slow background or boundary terms.
    'SIGMA_SB_WEAK_FACTOR': 0.005,  # Amplitude ratio for the weak boundary kernel; smaller values make the SB term weaker.
    'SB_MODE': 'off',  # Switch for the boundary/smooth-background kernel; off disables it and weak keeps only a weak background term.
    'ELL_INIT': 2.0,  # Initial length scale of the boundary kernel; larger values make the background term smoother.
    'ELL_LOWER': 0.5,  # Lower bound for the boundary-kernel length scale; prevents short-scale oscillations.
    'ELL_UPPER': 80.0,  # Upper bound for the boundary-kernel length scale; limits excessive smoothing of the background term.
    'SIGMA_L_INIT': 1e-12,  # Initial value of the linear/numerical-stability term, mainly used to keep the covariance matrix stable.
    'SIGMA_L_BOUNDS': (1e-14, 1e-9),  # Optimization bounds for the linear/numerical-stability term.
    'SIGMA_R_INIT_FACTOR': 0.10,  # Initial amplitude ratio of the real-component constant-offset kernel in the reInput branch.
    'SIGMA_R_BOUNDS': (1e-6, 3.0),  # Optimization bounds of the real-component constant-offset kernel in the reInput branch.
    'R_INF_TOP_FRACTION': 0.08,        # reInput branch: use the real component as input and predict the imaginary component
    'SIGMA_N_INIT_FACTOR': 5e-3,  # Initial sigma_n value relative to the data scale.
    'BASE_SIGMA_N_FLOOR_FACTOR': 5e-3,  # Base observation-noise floor; raw and other conditions are adjusted from this baseline.
    'SPARSE_SIGMA_N_FLOOR_FACTOR': 2e-3,  # Observation-noise floor under the sparse condition.
    'LIMITED_SIGMA_N_FLOOR_FACTOR': 2e-3,  # Observation-noise floor under limited conditions; helps avoid overfitting truncated data.
    'SIGMA_N_UPPER_FACTOR': 0.20,  # Relative upper bound for observation-noise sigma_n.
    'TAU_MAX_DEFAULT': 1e2,  # Default upper bound of the DRT integral; determines the maximum representable relaxation time.
    'TAU_MAX_FACTOR': 10.0,  # Scale factor used to dynamically extend tau_max from the lowest observed frequency.
    'TAU_MAX_FACTOR_LIMITED': 10.0,  # Scale factor for dynamic tau_max extension under limited conditions.
    'USE_DYNAMIC_TAU_MAX': True,  # Whether to estimate tau_max automatically from the current input-frequency range.
    'USE_NOISY_ADAPTIVE_SIGMA': True,
    'NOISY_SIGMA_N_INIT_FACTOR': 0.60,  # Initial sigma_n value under noisy conditions relative to the data scale and noise level.
    'NOISY_SIGMA_N_FLOOR_FACTOR': 0.20,  # Observation-noise floor under noisy conditions, usually higher than that under the raw condition.
    'NOISY_ELL_INIT': 2.0,  # Initial boundary-kernel length scale under noisy conditions; provides a smoother starting point for noisy data.
    'NOISY_ELL_FLOOR': 1.5,  # Lower bound for the boundary-kernel length scale under noisy conditions; suppresses fitting noise as high-frequency oscillations.
    'USE_LIMITED_DYNAMIC_PRED_RANGE': False,  # Whether to dynamically shrink the prediction-frequency range under limited conditions.
    'LIMITED_PRED_MARGIN_DECADE': 0.30,  # Number of additional decades added to both sides of the dynamic prediction range under limited conditions.
    'APPLY_L_OFFSET_REINPUT': False,
    'SB_KER_TYPE': 'IQ',
    'PRINT_OPTIMIZATION_TRACE': False,
    'RUN_NELDER_MEAD_REFINEMENT': False,
    'OPT_XTOL': 1e-6,
    'OPT_FTOL': 1e-8,
    'OPT_MAXITER': 800,  # Maximum number of iterations for the hyperparameter optimizer.
}


### Cell 03 - Main Workflow Execution

This cell runs the main workflow or intermediate data-organization steps of the current notebook.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.


In [ ]:
freq_vec = np.logspace(np.log10(FREQ_RANGE[0]), np.log10(FREQ_RANGE[1]), num=N_FREQS, endpoint=True)
omega_vec = 2.*pi*freq_vec


### Cell 04 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.
- Main parameters: R_1, R_2.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.


In [ ]:
# Parameter and hyperparameter settings
R_s = 90.0       # R_inf is used for real-component baseline correction and adding the offset back
R_1 = 450.0
tau_1 = 3.0e-5
alpha_1 = 0.92
R_2 = 300.0
tau_2 = 3.0e-3
alpha_2 = 0.85


### Cell 05 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

- Function descriptions:
  - `cole_term(freq, R, tau, alpha)`: Takes frequency, Cole resistance amplitude, time constant, and dispersion exponent, and returns the complex impedance of one Cole relaxation branch.

In [ ]:
def cole_term(freq, R, tau, alpha):
    return R / (1 + (1j * 2 * np.pi * freq * tau) ** alpha)
Z_exact = R_s + cole_term(freq_vec, R_1, tau_1, alpha_1) + cole_term(freq_vec, R_2, tau_2, alpha_2)
truth_df = pd.DataFrame({
    'model': MODEL_NAME,
    'freq': freq_vec,
    're': Z_exact.real,
    'imag': Z_exact.imag,
})
truth_df.head()


### Cell 06 - Plotting and Figure Export

This cell plots Nyquist, Bode, or method-comparison figures and saves them with a consistent naming rule.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
axes[0].plot(Z_exact.real, -Z_exact.imag, '-', color='black', lw=3, label='exact')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel(r'$Z_{re}/\Omega$')
axes[0].set_ylabel(r'$-Z_{im}/\Omega$')
axes[0].legend(frameon=False)
axes[0].set_title('Nyquist')
axes[1].semilogx(freq_vec, Z_exact.real, '-', color='black', lw=3)
axes[1].set_xlabel('f / Hz')
axes[1].set_ylabel(r'$Z_{re}/\Omega$')
axes[1].set_title('Real part')
axes[2].semilogx(freq_vec, -Z_exact.imag, '-', color='black', lw=3)
axes[2].set_xlabel('f / Hz')
axes[2].set_ylabel(r'$-Z_{im}/\Omega$')
axes[2].set_title('Imaginary part')
fig.tight_layout()
fig.savefig(FIG_DIR / f'{MODEL_NAME}_exact_spectrum.png', dpi=300, bbox_inches='tight')
plt.show()


### Cell 07 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

- Function descriptions:
  - `ensure_dir(path)`: Takes a directory path, creates missing directories, and returns the same `Path` object for later table or figure output.
  - `safe_sheet_name(name)`: Takes a candidate sheet name and the set of used names, and returns a valid, unique Excel sheet name.
  - `pad_to_length(arr, n)`: Pads an array to a common length so results on different frequency grids can be written into one table.
  - `safe_sd_from_var(var, counter, key, tol)`: Takes a variance array, clips small negative numerical values to zero, and returns posterior standard deviations.
  - `make_ci_columns(df, prefix, mean_col, sigma_col)`: according tomean and standard deviationgenerate ±1σ, ±2σ, ±3σ credible intervalcolumn.
  - `complex_rel_l2(z_pred, z_true)`: Computes the relative L2 error between two complex impedance curves and returns a dimensionless error.
  - `nrmse(y_pred, y_true)`: Computes the normalized root-mean-square error for comparing curves with different magnitudes.
  - `interp_complex(freq_source, z_source, freq_target)`: Interpolates a complex impedance curve onto a target frequency grid and returns the aligned complex array.
  - `degrade_noisy(freq, z, noise_level, seed)`: Applies multiplicative relative noise to a complex impedance spectrum and returns the noisy input spectrum.
  - `degrade_sparse(freq, z, ratio)`: Subsamples frequency points at a fixed interval and returns the sparse input spectrum.
  - `degrade_limited_range(freq, z, f_low, f_high)`: Keeps data within a specified frequency range and returns the limited-range input spectrum.
  - `degrade_limited_percent(freq, z, percent)`: Truncates points from both low- and high-frequency ends by a specified fraction and returns the limited-percent input spectrum.
  - `as_case_df(model_name, condition, freq, z)`: Combines frequency and complex impedance into a standard DataFrame containing freq, re, imag, and -imag columns.
  - `choose_prediction_grid(freq_train, pred_range, n_pred, condition, limited_dynamic, margin_decade)`: Builds the GP-HT prediction-frequency grid from the training frequencies, global prediction range, and input condition.
  - `infer_noise_level(condition)`: Reads the noisy-condition label and returns the corresponding relative noise level.
  - `tau_max_from_freq(freq_vec, default_tau_max, use_dynamic, factor)`: Estimates the DRT upper time bound tau_max from the lowest observed frequency.
  - `estimate_R_inf(freq_vec, z_re, top_fraction)`: Estimates R_inf from the highest-frequency real-component values for baseline correction.
  - `_condition_sigma_floor_factor(condition, noise_level, config)`: according to raw/noisy/sparse/limited conditionselectobservednoiselower boundratio.
  - `_make_kernel_options(std_scaled, sigma_SB0, ell0, tau_max, sb_on, config)`: Builds the GP-HT kernel-parameter dictionary from the data scale, condition, and global configuration.
  - `optimize_one_side(freq_vec, y_vec, side, config, condition)`: Optimizes GP-HT hyperparameters for one input component and returns the optimal parameters, training covariance, and optimization log.
  - `_build_training_factor(freq_vec, y_vec_physical, opt, side)`: Builds the Cholesky factor of the training covariance matrix for posterior prediction.
  - `_cho_solve_from_L(L, b)`: Solves a linear system with a Cholesky factor and returns the weights used in GP conditional means.
  - `predict_same_side(freq_train, y_train_physical, freq_pred, opt, side, counter)`: Performs GP regression for the input-side component and returns its posterior mean and standard deviation.
  - `predict_cross_im_to_re(freq_train, z_re_train, z_im_train, freq_pred, opt_im, counter)`: Uses the imaginary component as input and predicts the real component through the Hilbert cross-kernel.
  - `predict_cross_re_to_im(freq_train, z_re_centered_train, z_im_train, freq_pred, opt_re, counter, apply_L_offset)`: Uses the centered real component as input and predicts the imaginary component through the Hilbert cross-kernel.
  - `run_dual_gpht_case(model_name, condition, freq_train, z_train, freq_pred, config)`: Runs both imInput and reInput GP-HT branches for one degraded condition and returns predictions, credible intervals, and logs.
  - `plot_truth_and_degraded(model_name, truth_df, degraded_cases, out_dir)`: Plots the drift-free truth and degraded input spectra to check the generated noise, sparsity, or truncation condition.
  - `save_input_excel(model_name, truth_df, degraded_cases, data_dir)`: Saves the synthetic truth and degraded input spectra for later parameter recovery or external analysis.
  - `compute_metrics_for_case(truth_df, result_df)`: Computes curve-error metrics for degraded, imInput, and reInput spectra relative to the known truth.
  - `plot_regression_results(model_name, condition, freq_train, z_train, freq_pred, result_df, out_dir)`: Plots input-side GP regression results and credible intervals.
  - `plot_dual_gpht_results(model_name, condition, truth_df, input_df, result_df, out_dir)`: Plots the dual-branch GP-HT comparison between imInput and reInput outputs.
  - `save_results_excel(model_name, truth_df, degraded_cases, all_results, all_logs, all_metrics, data_dir)`: Exports input spectra, prediction spectra, credible intervals, logs, and metrics for all degraded conditions to Excel.

In [ ]:
# Key settings for the current workflow:
# Parameter and hyperparameter settings
# Noise-condition settings
# Scale the input magnitude to improve numerical conditioning
# GP-HT regression settings
# Cholesky decomposition for Gaussian-process linear algebra
# Posterior mean and covariance calculation
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path
def safe_sheet_name(name):
    """Excel sheet names must be <=31 characters and cannot contain special characters."""
    name = re.sub(r"[\[\]\:\*\?\/\\]", "_", str(name))
    return name[:31]
def pad_to_length(arr, n):
    arr = np.asarray(arr)
    if len(arr) >= n:
        return arr[:n]
    return np.pad(arr, (0, n - len(arr)), constant_values=np.nan)
def safe_sd_from_var(var, counter=None, key="neg_var", tol=1e-9):
    """
    Convert posterior variance to standard deviation.
    Do not use sqrt(abs(var)), because that hides numerical/pathological negative variances.
    """
    v = float(np.asarray(var).reshape(-1)[0])
    if counter is not None and v < -tol:
        counter[key] = counter.get(key, 0) + 1
    return float(np.sqrt(max(v, 0.0)))
def make_ci_columns(df, prefix, mean_col, sigma_col):
    """Add 1σ/2σ/3σ lower and upper credible-band columns."""
    for k in [1, 2, 3]:
        df[f"{prefix}_lower_{k}sigma"] = df[mean_col] - k * df[sigma_col]
        df[f"{prefix}_upper_{k}sigma"] = df[mean_col] + k * df[sigma_col]
    return df
def complex_rel_l2(z_pred, z_true):
    return float(np.linalg.norm(z_pred - z_true) / (np.linalg.norm(z_true) + 1e-30))
def nrmse(y_pred, y_true):
    denom = np.max(y_true) - np.min(y_true)
    if abs(denom) < 1e-30:
        denom = np.std(y_true) + 1e-30
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)) / denom)
def interp_complex(freq_source, z_source, freq_target):
    """Log-frequency interpolation of a complex spectrum."""
    x_src = np.log10(freq_source)
    x_tgt = np.log10(freq_target)
    re = np.interp(x_tgt, x_src, np.real(z_source))
    im = np.interp(x_tgt, x_src, np.imag(z_source))
    return re + 1j * im
def degrade_noisy(freq, z, noise_level, seed):
    """
    Multiplicative relative noise applied independently to Re and Im parts.
    This mimics percent-level measurement noise but is not exactly KK-consistent.
    """
    rng = np.random.default_rng(seed)
    noise_re = rng.normal(0.0, noise_level, size=len(freq))
    noise_im = rng.normal(0.0, noise_level, size=len(freq))
    z_noisy = z.real * (1.0 + noise_re) + 1j * z.imag * (1.0 + noise_im)
    return freq.copy(), z_noisy
def degrade_sparse(freq, z, ratio):
    """Keep one point every `ratio` points; always include the last frequency point."""
    idx = np.arange(0, len(freq), ratio)
    if idx[-1] != len(freq) - 1:
        idx = np.r_[idx, len(freq) - 1]
    idx = np.unique(idx)
    return freq[idx], z[idx]
def degrade_limited_range(freq, z, f_low, f_high):
    """Keep data only inside a prescribed frequency window."""
    mask = (freq >= f_low) & (freq <= f_high)
    return freq[mask], z[mask]
def degrade_limited_percent(freq, z, percent):
    """Remove percent/2 from both low-frequency and high-frequency ends."""
    percent = float(percent)
    n = len(freq)
    n_cut_each = int(round(n * percent / 2.0))
    if n_cut_each <= 0:
        return freq.copy(), z.copy()
    return freq[n_cut_each:n - n_cut_each], z[n_cut_each:n - n_cut_each]
def as_case_df(model_name, condition, freq, z):
    return pd.DataFrame({
        "model": model_name,
        "condition": condition,
        "freq": np.asarray(freq, dtype=float),
        "re": np.real(z).astype(float),
        "imag": np.imag(z).astype(float),
    })
def choose_prediction_grid(freq_train, pred_range, n_pred, condition, limited_dynamic=False, margin_decade=0.5):
    """
    Choose GP prediction grid.
    For limited data, dynamic mode avoids forcing a very long extrapolation interval.
    """
    condition = str(condition)
    if limited_dynamic and condition.startswith("limited"):
        f_min = max(pred_range[0], np.min(freq_train) / (10 ** margin_decade))
        f_max = min(pred_range[1], np.max(freq_train) * (10 ** margin_decade))
    else:
        f_min, f_max = pred_range
    return np.logspace(np.log10(f_min), np.log10(f_max), n_pred)
def infer_noise_level(condition):
    """Parse condition names such as noisy_0p1 -> 0.1."""
    condition = str(condition)
    if not condition.startswith("noisy"):
        return 0.0
    token = condition.replace("noisy_", "").replace("p", ".")
    try:
        return float(token)
    except Exception:
        return 0.0
def tau_max_from_freq(freq_vec, default_tau_max, use_dynamic, factor):
    if not use_dynamic:
        return float(default_tau_max)
    f_min = max(float(np.min(freq_vec)), 1e-300)
    return float(factor / (2.0 * np.pi * f_min))
def estimate_R_inf(freq_vec, z_re, top_fraction=0.08):
    """
    Estimate R_inf from the highest-frequency part of Re(Z).
    This is used only for the reInput branch, because HT acts on Z_re - R_inf.
    """
    freq_vec = np.asarray(freq_vec, dtype=float)
    z_re = np.asarray(z_re, dtype=float)
    n_top = max(3, int(np.ceil(len(freq_vec) * top_fraction)))
    idx = np.argsort(freq_vec)[-n_top:]
    return float(np.median(z_re[idx]))
def _condition_sigma_floor_factor(condition, noise_level, config):
    """Return the condition-dependent sigma_n floor factor relative to the current y scale."""
    c = str(condition)
    floor_factor = config.get("BASE_SIGMA_N_FLOOR_FACTOR", 5e-3)
    if c.startswith("sparse"):
        floor_factor = max(floor_factor, config.get("SPARSE_SIGMA_N_FLOOR_FACTOR", floor_factor))
    if c.startswith("limited"):
        floor_factor = max(floor_factor, config.get("LIMITED_SIGMA_N_FLOOR_FACTOR", floor_factor))
    if noise_level > 0 and config.get("USE_NOISY_ADAPTIVE_SIGMA", True):
        floor_factor = max(floor_factor, config.get("NOISY_SIGMA_N_FLOOR_FACTOR", 0.15) * noise_level)
    return float(floor_factor)
def _make_kernel_options(std_scaled, sigma_SB0, ell0, tau_max, sb_on, config):
    return {
        "sigma_DRT": config["SIGMA_DRT_FACTOR"] * std_scaled,
        "sigma_SB": sigma_SB0,
        "ell": ell0,
        "tau_max": tau_max,
        "DRT": True,
        "SB": sb_on,
        "SB_ker_type": config["SB_KER_TYPE"],
    }
def optimize_one_side(freq_vec, y_vec, side, config, condition="raw"):
    """
    Stabilized one-sided GP-HT hyperparameter optimization.
    side='im': y is Im(Z); theta=[sigma_n, sigma_DRT, sigma_SB, ell, sigma_L]
    side='re': y is Re(Z)-R_inf; theta=[sigma_n, sigma_DRT, sigma_SB, ell, sigma_R]
    Implementation notes:
    - y is scaled before optimization;
    - parameters are optimized in log-space;
    - L-BFGS-B bounds prevent sigma_n and ell from collapsing to zero;
    - sigma_n floors are also applied to raw/sparse/limited data, not only noisy data.
    """
    freq_vec = np.asarray(freq_vec, dtype=float)
    y_vec = np.asarray(y_vec, dtype=float)
    omega_vec = 2.0 * np.pi * freq_vec
    noise_level = infer_noise_level(condition)
    # Scale the input magnitude to improve numerical conditioning
    y_scale = float(np.median(np.abs(y_vec)))
    if not np.isfinite(y_scale) or y_scale < 1e-30:
        y_scale = float(np.std(y_vec) + 1e-30)
    y_scaled = y_vec / y_scale
    std_scaled = float(np.std(y_scaled) + 1e-30)
    med_abs_scaled = float(np.median(np.abs(y_scaled)) + 1e-30)
    tau_factor = config["TAU_MAX_FACTOR_LIMITED"] if str(condition).startswith("limited") else config["TAU_MAX_FACTOR"]
    tau_max = tau_max_from_freq(freq_vec, config["TAU_MAX_DEFAULT"], config["USE_DYNAMIC_TAU_MAX"], tau_factor)
    # Kernel-function calculation
    if config["SB_MODE"] == "off":
        sb_on = False
        sigma_SB0 = 1e-12
        sigma_SB_bounds = (1e-12, 1e-10)
    elif config["SB_MODE"] == "weak":
        sb_on = True
        sigma_SB0 = max(config["SIGMA_SB_WEAK_FACTOR"] * std_scaled, 1e-12)
        sigma_SB_bounds = (1e-12, max(0.10 * std_scaled, sigma_SB0 * 5.0, 1e-12))
    else:
        sb_on = True
        sigma_SB0 = max(config["SIGMA_SB_INIT_FACTOR"] * std_scaled, 1e-12)
        sigma_SB_bounds = (1e-12, max(0.30 * std_scaled, sigma_SB0 * 5.0, 1e-12))
    ell0 = float(config["ELL_INIT"])
    if noise_level > 0 and config.get("USE_NOISY_ADAPTIVE_SIGMA", True):
        ell0 = max(ell0, float(config.get("NOISY_ELL_INIT", ell0)))
    ell_lower = float(config.get("ELL_LOWER", 0.5))
    if noise_level > 0:
        ell_lower = max(ell_lower, float(config.get("NOISY_ELL_FLOOR", ell_lower)))
    ell_upper = float(config.get("ELL_UPPER", 100.0))
    ell0 = min(max(ell0, ell_lower), ell_upper)
    sigma_n_floor_factor = _condition_sigma_floor_factor(condition, noise_level, config)
    sigma_n_lower = max(sigma_n_floor_factor * med_abs_scaled, 1e-8)
    sigma_n0 = max(config["SIGMA_N_INIT_FACTOR"] * med_abs_scaled, sigma_n_lower * 1.2)
    if noise_level > 0 and config.get("USE_NOISY_ADAPTIVE_SIGMA", True):
        sigma_n0 = max(sigma_n0, config["NOISY_SIGMA_N_INIT_FACTOR"] * noise_level * med_abs_scaled)
    sigma_n_upper = max(float(config.get("SIGMA_N_UPPER_FACTOR", 2.0)) * med_abs_scaled, sigma_n_lower * 5.0)
    sigma_DRT0 = max(config["THETA_SIGMA_DRT_FACTOR"] * std_scaled, 1e-6)
    sigma_DRT_lower = max(float(config.get("SIGMA_DRT_LOWER_FACTOR", 1e-5)) * std_scaled, 1e-8)
    sigma_DRT_upper = max(float(config.get("SIGMA_DRT_UPPER_FACTOR", 10.0)) * std_scaled, sigma_DRT0 * 5.0)
    if side == "im":
        sigma_offset0 = max(float(config.get("SIGMA_L_INIT", 1e-12)), 1e-14)
        sigma_offset_bounds = tuple(config.get("SIGMA_L_BOUNDS", (1e-14, 1e-7)))
        type_data = "im"
    else:
        sigma_offset0 = max(config["SIGMA_R_INIT_FACTOR"] * std_scaled, 1e-6)
        sigma_offset_bounds = tuple(config.get("SIGMA_R_BOUNDS", (1e-6, 5.0)))
        type_data = "re"
    theta_0 = np.array([sigma_n0, sigma_DRT0, sigma_SB0, ell0, sigma_offset0], dtype=float)
    bounds = [
        (sigma_n_lower, sigma_n_upper),
        (sigma_DRT_lower, sigma_DRT_upper),
        sigma_SB_bounds,
        (ell_lower, ell_upper),
        sigma_offset_bounds,
    ]
    theta_0 = np.array([min(max(v, lo * 1.0001), hi / 1.0001) for v, (lo, hi) in zip(theta_0, bounds)], dtype=float)
    log_bounds = [(np.log(lo), np.log(hi)) for lo, hi in bounds]
    eta0 = np.log(theta_0)
    ker_opts = _make_kernel_options(std_scaled, sigma_SB0, ell0, tau_max, sb_on, config)
    def objective_log(eta):
        theta = np.exp(eta)
        if not np.all(np.isfinite(theta)):
            return 1e300
        try:
            return float(gpf.NMLL_fct(theta, y_scaled, omega_vec, ker_opts, type_data))
        except Exception:
            return 1e300
    res = minimize(
        objective_log,
        eta0,
        method="L-BFGS-B",
        bounds=log_bounds,
        options={"maxiter": int(config.get("OPT_MAXITER", 500)), "ftol": float(config.get("OPT_FTOL", 1e-8))},
    )
    theta_opt = np.exp(res.x)
    if config.get("RUN_NELDER_MEAD_REFINEMENT", False):
        lo = np.array([b[0] for b in log_bounds])
        hi = np.array([b[1] for b in log_bounds])
        def nm_obj(eta):
            if np.any(eta < lo) or np.any(eta > hi):
                return 1e300 + float(np.sum((np.minimum(0, eta - lo))**2 + (np.maximum(0, eta - hi))**2))
            return objective_log(eta)
        res_nm = minimize(
            nm_obj,
            res.x,
            method="Nelder-Mead",
            options={"maxiter": int(config.get("OPT_MAXITER", 500)),
                     "xatol": float(config.get("OPT_XTOL", 1e-6)),
                     "fatol": float(config.get("OPT_FTOL", 1e-6))},
        )
        if res_nm.fun < res.fun:
            res = res_nm
            theta_opt = np.exp(res_nm.x)
    ker_opts_opt = ker_opts.copy()
    ker_opts_opt["sigma_DRT"] = theta_opt[1]
    ker_opts_opt["sigma_SB"] = theta_opt[2]
    ker_opts_opt["ell"] = theta_opt[3]
    return {
        "theta": theta_opt,
        "theta0": theta_0,
        "bounds": bounds,
        "ker_opts": ker_opts_opt,
        "tau_max": tau_max,
        "side": side,
        "type_data": type_data,
        "y_scale": y_scale,
        "y_train_scaled": y_scaled,
        "success": bool(getattr(res, "success", False)),
        "message": str(getattr(res, "message", "")),
        "nll": float(getattr(res, "fun", np.nan)),
        "sigma_n_floor_scaled": sigma_n_lower,
        "sigma_n_floor_physical": sigma_n_lower * y_scale,
        "sigma_n_physical": theta_opt[0] * y_scale,
    }
def _build_training_factor(freq_vec, y_vec_physical, opt, side):
    """Build Cholesky factor and alpha=K^{-1}y without explicitly forming K^{-1}."""
    omega_vec = 2.0 * np.pi * np.asarray(freq_vec, dtype=float)
    y_scaled = np.asarray(y_vec_physical, dtype=float) / opt["y_scale"]
    theta = opt["theta"]
    sigma_n = theta[0]
    ker_opts = opt["ker_opts"]
    n = len(freq_vec)
    if side == "im":
        sigma_L = theta[4]
        K = gpf.mat_K(omega_vec, omega_vec, ker_opts, "im")
        K_full = K + sigma_n**2 * np.eye(n) + sigma_L**2 * np.outer(omega_vec, omega_vec)
    else:
        sigma_R = theta[4]
        K = gpf.mat_K(omega_vec, omega_vec, ker_opts, "re")
        K_full = K + sigma_n**2 * np.eye(n) + sigma_R**2 * np.ones((n, n))
    # Cholesky decomposition for Gaussian-process linear algebra
    jitter = max(float(opt.get("sigma_n_floor_scaled", 1e-8))**2 * 1e-4, 1e-12)
    eye = np.eye(n)
    for _ in range(8):
        try:
            L = np.linalg.cholesky(K_full + jitter * eye)
            break
        except np.linalg.LinAlgError:
            jitter *= 10.0
    else:
        K_full = gpf.nearest_PD(K_full + jitter * eye)
        L = np.linalg.cholesky(K_full)
    v = np.linalg.solve(L, y_scaled)
    alpha = np.linalg.solve(L.T, v)
    return omega_vec, L, alpha
def _cho_solve_from_L(L, b):
    """Solve (L L.T) x = b."""
    v = np.linalg.solve(L, b)
    return np.linalg.solve(L.T, v)
def predict_same_side(freq_train, y_train_physical, freq_pred, opt, side, counter=None):
    """Bayesian GP regression of the same side: im->im or centered re->centered re."""
    omega_train, L, alpha = _build_training_factor(freq_train, y_train_physical, opt, side)
    omega_pred = 2.0 * np.pi * np.asarray(freq_pred, dtype=float)
    theta = opt["theta"]
    ker_opts = opt["ker_opts"]
    y_scale = opt["y_scale"]
    mu = np.zeros_like(freq_pred, dtype=float)
    sd = np.zeros_like(freq_pred, dtype=float)
    for i, w in enumerate(omega_pred):
        w_np = np.array([w])
        if side == "im":
            sigma_L = theta[4]
            k = gpf.mat_K(omega_train, w_np, ker_opts, "im").flatten() + sigma_L**2 * omega_train * w
            kss = gpf.mat_K(w_np, w_np, ker_opts, "im").flatten()[0] + sigma_L**2 * w**2
        else:
            sigma_R = theta[4]
            k = gpf.mat_K(omega_train, w_np, ker_opts, "re").flatten() + sigma_R**2 * np.ones(len(freq_train))
            kss = gpf.mat_K(w_np, w_np, ker_opts, "re").flatten()[0] + sigma_R**2
        mu_scaled = k @ alpha
        v = _cho_solve_from_L(L, k)
        var_scaled = kss - k @ v
        mu[i] = mu_scaled * y_scale
        sd[i] = safe_sd_from_var(var_scaled, counter, f"neg_var_reg_{side}") * y_scale
    return mu, sd
def predict_cross_im_to_re(freq_train, z_re_train, z_im_train, freq_pred, opt_im, counter=None):
    """Input Im(Z) -> predict Re(Z). R_inf/constant offset is aligned on training data."""
    omega_train, L, alpha = _build_training_factor(freq_train, z_im_train, opt_im, "im")
    omega_pred = 2.0 * np.pi * np.asarray(freq_pred, dtype=float)
    theta = opt_im["theta"]
    ker_opts = opt_im["ker_opts"]
    y_scale = opt_im["y_scale"]
    def raw_pred(freq_target):
        omega_target = 2.0 * np.pi * np.asarray(freq_target, dtype=float)
        out = np.zeros_like(freq_target, dtype=float)
        for j, w in enumerate(omega_target):
            k = gpf.mat_K(omega_train, np.array([w]), ker_opts, "im-re").flatten()
            out[j] = (k @ alpha) * y_scale
        return out
    mu_train_raw = raw_pred(freq_train)
    R_offset = float(np.median(np.asarray(z_re_train, dtype=float) - mu_train_raw))
    mu = np.zeros_like(freq_pred, dtype=float)
    sd = np.zeros_like(freq_pred, dtype=float)
    for i, w in enumerate(omega_pred):
        k = gpf.mat_K(omega_train, np.array([w]), ker_opts, "im-re").flatten()
        kss = gpf.mat_K(np.array([w]), np.array([w]), ker_opts, "re").flatten()[0]
        mu_scaled = k @ alpha
        v = _cho_solve_from_L(L, k)
        var_scaled = kss - k @ v
        mu[i] = R_offset + mu_scaled * y_scale
        sd[i] = safe_sd_from_var(var_scaled, counter, "neg_var_ht_im_to_re") * y_scale
    return mu, sd, R_offset
def predict_cross_re_to_im(freq_train, z_re_centered_train, z_im_train, freq_pred, opt_re, counter=None, apply_L_offset=False):
    """Input centered Re(Z)-R_inf -> predict Im(Z)."""
    omega_train, L, alpha = _build_training_factor(freq_train, z_re_centered_train, opt_re, "re")
    omega_pred = 2.0 * np.pi * np.asarray(freq_pred, dtype=float)
    ker_opts = opt_re["ker_opts"]
    y_scale = opt_re["y_scale"]
    def raw_pred(freq_target):
        omega_target = 2.0 * np.pi * np.asarray(freq_target, dtype=float)
        out = np.zeros_like(freq_target, dtype=float)
        for j, w in enumerate(omega_target):
            k = gpf.mat_K(omega_train, np.array([w]), ker_opts, "re-im").flatten()
            out[j] = (k @ alpha) * y_scale
        return out
    mu_train_raw = raw_pred(freq_train)
    if apply_L_offset:
        denom = float(np.dot(omega_train, omega_train)) + 1e-30
        L_offset = float(np.dot(omega_train, np.asarray(z_im_train) - mu_train_raw) / denom)
    else:
        L_offset = 0.0
    mu = np.zeros_like(freq_pred, dtype=float)
    sd = np.zeros_like(freq_pred, dtype=float)
    for i, w in enumerate(omega_pred):
        k = gpf.mat_K(omega_train, np.array([w]), ker_opts, "re-im").flatten()
        kss = gpf.mat_K(np.array([w]), np.array([w]), ker_opts, "im").flatten()[0]
        mu_scaled = k @ alpha
        v = _cho_solve_from_L(L, k)
        var_scaled = kss - k @ v
        mu[i] = mu_scaled * y_scale + w * L_offset
        sd[i] = safe_sd_from_var(var_scaled, counter, "neg_var_ht_re_to_im") * y_scale
    return mu, sd, L_offset
def run_dual_gpht_case(model_name, condition, freq_train, z_train, freq_pred, config):
    """
    Run two one-sided GP-HT branches:
    imInput: input Im(Z), predict Re(Z), regress Im(Z).
    reInput: input centered Re(Z)-R_inf, predict Im(Z), and regress centered Re(Z) before adding R_inf back.
    """
    z_re = np.real(z_train).astype(float)
    z_im = np.imag(z_train).astype(float)
    counter = {}
    # R_inf is used for real-component baseline correction and adding the offset back
    R_inf_hat = estimate_R_inf(freq_train, z_re, top_fraction=config.get("R_INF_TOP_FRACTION", 0.08))
    z_re_centered = z_re - R_inf_hat
    opt_im = optimize_one_side(freq_train, z_im, "im", config, condition)
    opt_re = optimize_one_side(freq_train, z_re_centered, "re", config, condition)
    im_reg, im_reg_sd = predict_same_side(freq_train, z_im, freq_pred, opt_im, "im", counter)
    re_from_im, re_from_im_sd, R_offset = predict_cross_im_to_re(freq_train, z_re, z_im, freq_pred, opt_im, counter)
    re_centered_reg, re_centered_reg_sd = predict_same_side(freq_train, z_re_centered, freq_pred, opt_re, "re", counter)
    re_reg = re_centered_reg + R_inf_hat
    re_reg_sd = re_centered_reg_sd
    im_from_re, im_from_re_sd, L_offset = predict_cross_re_to_im(
        freq_train, z_re_centered, z_im, freq_pred, opt_re, counter,
        apply_L_offset=config.get("APPLY_L_OFFSET_REINPUT", False)
    )
    df = pd.DataFrame({
        "model": model_name,
        "condition": condition,
        "freq_pred": freq_pred,
        # GP-HT regression settings
        "imInput_re_pred": re_from_im,
        "imInput_imag_reg": im_reg,
        "imInput_var_re_pred": re_from_im_sd**2,
        "imInput_var_imag_reg": im_reg_sd**2,
        "imInput_sigma_re_pred": re_from_im_sd,
        "imInput_sigma_imag_reg": im_reg_sd,
        # Input-branch description
        "reInput_Rinf_hat": R_inf_hat,
        "reInput_re_centered_reg": re_centered_reg,
        "reInput_re_reg": re_reg,
        "reInput_imag_pred": im_from_re,
        "reInput_var_re_reg": re_reg_sd**2,
        "reInput_var_imag_pred": im_from_re_sd**2,
        "reInput_sigma_re_reg": re_reg_sd,
        "reInput_sigma_imag_pred": im_from_re_sd,
        # Input-branch description
        "gp_re": re_from_im,
        "gp_imag": im_reg,
        "gp_sigma_re": re_from_im_sd,
        "gp_sigma_imag": im_reg_sd,
    })
    for prefix, mean_col, sigma_col in [
        ("imInput_re_pred", "imInput_re_pred", "imInput_sigma_re_pred"),
        ("imInput_imag_reg", "imInput_imag_reg", "imInput_sigma_imag_reg"),
        ("reInput_re_reg", "reInput_re_reg", "reInput_sigma_re_reg"),
        ("reInput_imag_pred", "reInput_imag_pred", "reInput_sigma_imag_pred"),
        ("gp_re", "gp_re", "gp_sigma_re"),
        ("gp_imag", "gp_imag", "gp_sigma_imag"),
    ]:
        df = make_ci_columns(df, prefix, mean_col, sigma_col)
    log = {
        "model": model_name,
        "condition": condition,
        "n_train": len(freq_train),
        "f_min_train": float(np.min(freq_train)),
        "f_max_train": float(np.max(freq_train)),
        "R_inf_hat_reInput": float(R_inf_hat),
        "R_offset_imInput": float(R_offset),
        "L_offset_reInput": float(L_offset),
        # Scale the input magnitude to improve numerical conditioning
        "theta_im_sigma_n_scaled": float(opt_im["theta"][0]),
        "theta_im_sigma_n_physical": float(opt_im["sigma_n_physical"]),
        "theta_im_sigma_DRT": float(opt_im["theta"][1]),
        "theta_im_sigma_SB": float(opt_im["theta"][2]),
        "theta_im_ell": float(opt_im["theta"][3]),
        "theta_im_sigma_L": float(opt_im["theta"][4]),
        "theta_im_y_scale": float(opt_im["y_scale"]),
        "theta_im_sigma_n_floor_physical": float(opt_im["sigma_n_floor_physical"]),
        "tau_max_im": float(opt_im["tau_max"]),
        "nll_im": float(opt_im["nll"]),
        "success_im": bool(opt_im["success"]),
        "theta_re_sigma_n_scaled": float(opt_re["theta"][0]),
        "theta_re_sigma_n_physical": float(opt_re["sigma_n_physical"]),
        "theta_re_sigma_DRT": float(opt_re["theta"][1]),
        "theta_re_sigma_SB": float(opt_re["theta"][2]),
        "theta_re_ell": float(opt_re["theta"][3]),
        "theta_re_sigma_R": float(opt_re["theta"][4]),
        "theta_re_y_scale": float(opt_re["y_scale"]),
        "theta_re_sigma_n_floor_physical": float(opt_re["sigma_n_floor_physical"]),
        "tau_max_re": float(opt_re["tau_max"]),
        "nll_re": float(opt_re["nll"]),
        "success_re": bool(opt_re["success"]),
    }
    for key, value in counter.items():
        log[key] = int(value)
    return df, log, opt_im, opt_re
def plot_truth_and_degraded(model_name, truth_df, degraded_cases, out_dir):
    """Plot exact truth and all degraded inputs."""
    out_dir = ensure_dir(out_dir)
    n_cases = len(degraded_cases)
    fig, axes = plt.subplots(n_cases, 3, figsize=(15, 4.0 * n_cases))
    if n_cases == 1:
        axes = np.array([axes])
    f_truth = truth_df["freq"].values
    z_truth = truth_df["re"].values + 1j * truth_df["imag"].values
    for row, (condition, df) in enumerate(degraded_cases.items()):
        f = df["freq"].values
        z = df["re"].values + 1j * df["imag"].values
        ax = axes[row, 0]
        ax.plot(z_truth.real, -z_truth.imag, "-", color="black", lw=2.5, label="Exact truth")
        ax.scatter(z.real, -z.imag, s=18, color="tab:red", label="Input/degraded")
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(r"$Z_{re}/\Omega$")
        ax.set_ylabel(r"$-Z_{im}/\Omega$")
        ax.set_title(f"{condition} - Nyquist")
        ax.legend(frameon=False, fontsize=9)
        ax = axes[row, 1]
        ax.semilogx(f_truth, z_truth.real, "-", color="black", lw=2.5, label="Exact truth")
        ax.scatter(f, z.real, s=18, color="tab:red", label="Input/degraded")
        ax.set_xlabel("f / Hz")
        ax.set_ylabel(r"$Z_{re}/\Omega$")
        ax.set_title("Real part")
        ax.legend(frameon=False, fontsize=9)
        ax = axes[row, 2]
        ax.semilogx(f_truth, -z_truth.imag, "-", color="black", lw=2.5, label="Exact truth")
        ax.scatter(f, -z.imag, s=18, color="tab:red", label="Input/degraded")
        ax.set_xlabel("f / Hz")
        ax.set_ylabel(r"$-Z_{im}/\Omega$")
        ax.set_title("Imaginary part")
        ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    out_path = out_dir / f"{model_name}_truth_and_degraded.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved degraded overview figure to:", out_path)
def save_input_excel(model_name, truth_df, degraded_cases, data_dir):
    """Save exact truth and all degraded inputs. Each condition is written to a separate sheet."""
    data_dir = ensure_dir(data_dir)
    out_path = data_dir / f"{model_name}_truth_and_degraded_data.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        truth_df.to_excel(writer, sheet_name=safe_sheet_name("truth"), index=False)
        for condition, df in degraded_cases.items():
            df.to_excel(writer, sheet_name=safe_sheet_name(condition), index=False)
    return out_path
def compute_metrics_for_case(truth_df, result_df):
    """Compute reconstruction metrics against exact synthetic truth."""
    f_pred = result_df["freq_pred"].values
    z_truth_full = truth_df["re"].values + 1j * truth_df["imag"].values
    z_truth_pred = interp_complex(truth_df["freq"].values, z_truth_full, f_pred)
    z_imInput = result_df["imInput_re_pred"].values + 1j * result_df["imInput_imag_reg"].values
    z_reInput = result_df["reInput_re_reg"].values + 1j * result_df["reInput_imag_pred"].values
    def r2_complex(z_hat, z_true):
        ss_res = np.sum(np.abs(z_hat - z_true) ** 2)
        z_mean = np.mean(z_true)
        ss_tot = np.sum(np.abs(z_true - z_mean) ** 2) + 1e-30
        return float(1 - ss_res / ss_tot)
    return {
        "imInput_complex_rel_l2": complex_rel_l2(z_imInput, z_truth_pred),
        "reInput_complex_rel_l2": complex_rel_l2(z_reInput, z_truth_pred),
        "imInput_nrmse_re": nrmse(z_imInput.real, z_truth_pred.real),
        "imInput_nrmse_imag": nrmse(z_imInput.imag, z_truth_pred.imag),
        "reInput_nrmse_re": nrmse(z_reInput.real, z_truth_pred.real),
        "reInput_nrmse_imag": nrmse(z_reInput.imag, z_truth_pred.imag),
        "imInput_r2_complex": r2_complex(z_imInput, z_truth_pred),
        "reInput_r2_complex": r2_complex(z_reInput, z_truth_pred),
    }
def plot_regression_results(model_name, condition, freq_train, z_train, freq_pred, result_df, out_dir):
    """Plot same-side GP regression results."""
    out_dir = ensure_dir(out_dir)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    ax = axes[0]
    y = result_df["reInput_re_reg"].values
    sd = result_df["reInput_sigma_re_reg"].values
    ax.scatter(freq_train, z_train.real, s=22, color="tab:red", label="Input real")
    ax.semilogx(freq_pred, y, "-", color="tab:green", lw=2.5, label="Real regression")
    ax.fill_between(freq_pred, y - 2 * sd, y + 2 * sd, color="tab:green", alpha=0.18, label=r"$\pm2\sigma$")
    ax.set_xlabel("f / Hz")
    ax.set_ylabel(r"$Z_{re}/\Omega$")
    ax.set_title(f"{condition}: real regression")
    ax.legend(frameon=False, fontsize=9)
    ax = axes[1]
    y_im = result_df["imInput_imag_reg"].values
    sd_im = result_df["imInput_sigma_imag_reg"].values
    ax.scatter(freq_train, -z_train.imag, s=22, color="tab:red", label="Input -imag")
    ax.semilogx(freq_pred, -y_im, "-", color="tab:blue", lw=2.5, label="Imag regression")
    ax.fill_between(freq_pred, -y_im - 2 * sd_im, -y_im + 2 * sd_im, color="tab:blue", alpha=0.18, label=r"$\pm2\sigma$")
    ax.set_xlabel("f / Hz")
    ax.set_ylabel(r"$-Z_{im}/\Omega$")
    ax.set_title(f"{condition}: imaginary regression")
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    out_path = out_dir / f"{model_name}_{condition}_regression.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
def plot_dual_gpht_results(model_name, condition, truth_df, input_df, result_df, out_dir):
    """Plot dual one-sided GP-HT prediction results."""
    out_dir = ensure_dir(out_dir)
    f_truth = truth_df["freq"].values
    z_truth = truth_df["re"].values + 1j * truth_df["imag"].values
    f_in = input_df["freq"].values
    z_in = input_df["re"].values + 1j * input_df["imag"].values
    f = result_df["freq_pred"].values
    z_imInput = result_df["imInput_re_pred"].values + 1j * result_df["imInput_imag_reg"].values
    sd_imInput_re = result_df["imInput_sigma_re_pred"].values
    sd_imInput_im = result_df["imInput_sigma_imag_reg"].values
    z_reInput = result_df["reInput_re_reg"].values + 1j * result_df["reInput_imag_pred"].values
    sd_reInput_re = result_df["reInput_sigma_re_reg"].values
    sd_reInput_im = result_df["reInput_sigma_imag_pred"].values
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    ax = axes[0]
    ax.plot(z_truth.real, -z_truth.imag, "-", color="black", lw=2.5, label="Exact truth")
    ax.scatter(z_in.real, -z_in.imag, s=18, color="tab:red", label="Input/degraded")
    ax.plot(z_imInput.real, -z_imInput.imag, "-", color="tab:blue", lw=2.3, label="imInput branch")
    ax.plot(z_reInput.real, -z_reInput.imag, "--", color="tab:green", lw=2.3, label="reInput branch")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$Z_{re}/\Omega$")
    ax.set_ylabel(r"$-Z_{im}/\Omega$")
    ax.set_title(f"{condition}: Nyquist")
    ax.legend(frameon=False, fontsize=8)
    ax = axes[1]
    ax.semilogx(f_truth, z_truth.real, "-", color="black", lw=2.5, label="Exact truth")
    ax.scatter(f_in, z_in.real, s=18, color="tab:red", label="Input/degraded")
    ax.semilogx(f, z_imInput.real, "-", color="tab:blue", lw=2.3, label="imInput: Re predicted")
    ax.fill_between(f, z_imInput.real - 2 * sd_imInput_re, z_imInput.real + 2 * sd_imInput_re, color="tab:blue", alpha=0.16)
    ax.semilogx(f, z_reInput.real, "--", color="tab:green", lw=2.3, label="reInput: Re regression")
    ax.fill_between(f, z_reInput.real - 2 * sd_reInput_re, z_reInput.real + 2 * sd_reInput_re, color="tab:green", alpha=0.12)
    ax.set_xlabel("f / Hz")
    ax.set_ylabel(r"$Z_{re}/\Omega$")
    ax.set_title("Real part")
    ax.legend(frameon=False, fontsize=8)
    ax = axes[2]
    ax.semilogx(f_truth, -z_truth.imag, "-", color="black", lw=2.5, label="Exact truth")
    ax.scatter(f_in, -z_in.imag, s=18, color="tab:red", label="Input/degraded")
    ax.semilogx(f, -z_imInput.imag, "-", color="tab:blue", lw=2.3, label="imInput: Imag regression")
    ax.fill_between(f, -z_imInput.imag - 2 * sd_imInput_im, -z_imInput.imag + 2 * sd_imInput_im, color="tab:blue", alpha=0.16)
    ax.semilogx(f, -z_reInput.imag, "--", color="tab:green", lw=2.3, label="reInput: Imag predicted")
    ax.fill_between(f, -z_reInput.imag - 2 * sd_reInput_im, -z_reInput.imag + 2 * sd_reInput_im, color="tab:green", alpha=0.12)
    ax.set_xlabel("f / Hz")
    ax.set_ylabel(r"$-Z_{im}/\Omega$")
    ax.set_title("Imaginary part")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    out_path = out_dir / f"{model_name}_{condition}_dual_GPHT.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
def save_results_excel(model_name, truth_df, degraded_cases, all_results, all_logs, all_metrics, data_dir):
    """Save all GP-HT dual-side results, logs, and metrics to one Excel file."""
    data_dir = ensure_dir(data_dir)
    out_path = data_dir / f"{model_name}_GPHT_dual_side_results.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        truth_df.to_excel(writer, sheet_name=safe_sheet_name("truth"), index=False)
        for condition, input_df in degraded_cases.items():
            input_df.to_excel(writer, sheet_name=safe_sheet_name("input_" + condition), index=False)
        for condition, result_df in all_results.items():
            result_df.to_excel(writer, sheet_name=safe_sheet_name("gpht_" + condition), index=False)
        pd.DataFrame(all_logs).to_excel(writer, sheet_name=safe_sheet_name("optimization_log"), index=False)
        pd.DataFrame(all_metrics).to_excel(writer, sheet_name=safe_sheet_name("metrics"), index=False)
    return out_path


### Cell 08 - Main Workflow Execution

This cell runs the main workflow or intermediate data-organization steps of the current notebook.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
# Noise-condition settings
degraded_cases = {}
raw_freq, raw_z = freq_vec.copy(), Z_exact.copy()
degraded_cases['raw'] = as_case_df(MODEL_NAME, 'raw', raw_freq, raw_z)
for noise_level in NOISE_LEVELS:
    f_noisy, z_noisy = degrade_noisy(freq_vec, Z_exact, noise_level, seed=RANDOM_SEED + int(noise_level*10000))
    condition = f"noisy_{str(noise_level).replace('.', 'p')}"
    degraded_cases[condition] = as_case_df(MODEL_NAME, condition, f_noisy, z_noisy)
list(degraded_cases.keys())


### Cell 09 - Main Workflow Execution

This cell runs the main workflow or intermediate data-organization steps of the current notebook.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
# Sparse-sampling condition settings
for ratio in SPARSE_RATIOS:
    f_sparse, z_sparse = degrade_sparse(freq_vec, Z_exact, ratio)
    condition = f"sparse_{ratio}"
    degraded_cases[condition] = as_case_df(MODEL_NAME, condition, f_sparse, z_sparse)
list(degraded_cases.keys())


### Cell 10 - Main Workflow Execution

This cell runs the main workflow or intermediate data-organization steps of the current notebook.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
# Frequency-limitation or endpoint-truncation settings
# Frequency range and number of points
for f_low, f_high in LIMITED_RANGE_LIST:
    f_lim, z_lim = degrade_limited_range(freq_vec, Z_exact, f_low, f_high)
    condition = f"limited_range_{f_low:.0e}_{f_high:.0e}"
    degraded_cases[condition] = as_case_df(MODEL_NAME, condition, f_lim, z_lim)
# Frequency-limitation or endpoint-truncation settings
for percent in LIMITED_PERCENT_LIST:
    f_lim, z_lim = degrade_limited_percent(freq_vec, Z_exact, percent)
    condition = f"limited_pct_{int(percent*100)}"
    degraded_cases[condition] = as_case_df(MODEL_NAME, condition, f_lim, z_lim)
list(degraded_cases.keys())


### Cell 11 - Plotting and Figure Export

This cell plots Nyquist, Bode, or method-comparison figures and saves them with a consistent naming rule.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
plot_truth_and_degraded(MODEL_NAME, truth_df, degraded_cases, DEG_FIG_DIR)


### Cell 12 - Read or Write Tabular Data

This cell reads input workbooks or writes computed results to Excel for later plotting, statistics, and checking.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
input_excel_path = save_input_excel(MODEL_NAME, truth_df, degraded_cases, DATA_DIR)
print('Saved input datasets to:', input_excel_path)


### Cell 13 - Plotting and Figure Export

This cell plots Nyquist, Bode, or method-comparison figures and saves them with a consistent naming rule.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
all_results = {}
all_logs = []
all_metrics = []
all_opt_im = {}
all_opt_re = {}
for condition, input_df in degraded_cases.items():
    print('\n===', condition, '===')
    freq_train = input_df['freq'].values.astype(float)
    z_train = input_df['re'].values.astype(float) + 1j*input_df['imag'].values.astype(float)
    freq_pred = choose_prediction_grid(
        freq_train,
        PRED_RANGE,
        N_PRED,
        condition,
        limited_dynamic=GPHT_CONFIG['USE_LIMITED_DYNAMIC_PRED_RANGE'],
        margin_decade=GPHT_CONFIG['LIMITED_PRED_MARGIN_DECADE'],
    )
    result_df, log, opt_im, opt_re = run_dual_gpht_case(MODEL_NAME, condition, freq_train, z_train, freq_pred, GPHT_CONFIG)
    all_results[condition] = result_df
    all_logs.append(log)
    all_opt_im[condition] = opt_im
    all_opt_re[condition] = opt_re
    metric = compute_metrics_for_case(truth_df, result_df)
    metric.update({'condition': condition})
    all_metrics.append(metric)
    plot_regression_results(MODEL_NAME, condition, freq_train, z_train, freq_pred, result_df, REG_FIG_DIR)
pd.DataFrame(all_logs)


### Cell 14 - Plotting and Figure Export

This cell plots Nyquist, Bode, or method-comparison figures and saves them with a consistent naming rule.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
# Plotting settings
# Input-branch description
# Input-branch description
for condition, result_df in all_results.items():
    plot_dual_gpht_results(MODEL_NAME, condition, truth_df, degraded_cases[condition], result_df, GPHT_FIG_DIR)
pd.DataFrame(all_metrics)


### Cell 15 - Read or Write Tabular Data

This cell reads input workbooks or writes computed results to Excel for later plotting, statistics, and checking.

- Purpose: Double Cole synthetic-spectrum generation, degraded-condition construction, and dual-branch GP-HT reconstruction of real and imaginary components.

In [ ]:
results_excel_path = save_results_excel(MODEL_NAME, truth_df, degraded_cases, all_results, all_logs, all_metrics, DATA_DIR)
print('Saved GP-HT dual-side results to:', results_excel_path)
# Save results
pd.DataFrame(all_logs).to_csv(DATA_DIR / f'{MODEL_NAME}_optimization_log.csv', index=False)
pd.DataFrame(all_metrics).to_csv(DATA_DIR / f'{MODEL_NAME}_metrics.csv', index=False)
# Save results
long_df = pd.concat(all_results.values(), ignore_index=True)
long_csv_path = DATA_DIR / f'{MODEL_NAME}_GPHT_dual_side_results_long.csv'
long_df.to_csv(long_csv_path, index=False)
print('Saved long-format results to:', long_csv_path)
